# Defining own plugins

A platform is a `Plugin`: what it can do, which secrets it needs, and those calls.
Real ones are packages (`uv run task plugin:new -- <platform> <Class> "<description>"`)
found through an entry point; here one is registered by hand, for this notebook.

In [1]:
import os
from amoorion import Amoorion

# Examples stay off your real setup: see Amoorion.init
Amoorion.init()


PosixPath('/tmp/amoorion-517ddlmt')

In [2]:
from datetime import UTC, datetime

from typing import ClassVar



from amoorion import Plugins

from amoorion.models import Message, Post

from amoorion.plugins import PLUGINS, Capability, Plugin



class Notebook(Plugin):

    name: ClassVar[str] = "notebook"
    
    description: ClassVar[str] = "prints posts in the notebook"
    
    capabilities: ClassVar[
        frozenset[ Capability ]
    ] = frozenset(
        {
            Capability.POST
        }
    )
    
    secrets: ClassVar[
        tuple[
            str, 
            ...
        ]
    ] = ("greeting",)

    def post(self, post: Post) -> Message:

        text = f"{self.secret('greeting')}: {post.text}"

        print(text)

        return Message(
            platform=self.name, 
            text=text, 
            id="1", 
            created_at=datetime.now(UTC)
        )



PLUGINS.register(Notebook)

Plugins["notebook"]

<Plug notebook unknown: post>

It now is like any other, secrets from the vault included:

In [3]:
print(
    Plugins["notebook"].infoflags
)

os.environ["AMOORION_SECRET_NOTEBOOK_GREETING"] = "hello"

print(
    Plugins["notebook"].infoflags["creds"]
)

Amoorion().use(
    Plugins["notebook"], 
    "dryrun"
).publish(
    text="from my own plugin"
)[0]["ok"]


{'creds': False, 'can_post': True, 'can_read': False, 'can_reply': False, 'can_delete': False}
True
hello: from my own plugin


True

What it can't do it refuses clearly:

In [4]:
try:
    Amoorion().read(
        platform="notebook"
    )

except Exception as error:
    print(
        type(error).__name__, 
        "-", 
        error
    )

Unsupported - notebook can't read


Test a real plugin against a fake of its platform's API, as every package in
`packages/` does. Guide: [docs/plugins.md](../docs/plugins.md).